# Bronze Raw Data Generator (CSV)

Generates synthetic raw source data (items, orders, products, suppliers, customers) for the upwind_demo bronze/silver/gold pipeline.

Each table is generated in **SQL** into a temp view, then exported as CSV files to the Unity Catalog volume `ofer_demo.bronze.raw_csv`, where the bronze layer of the Spark Declarative Pipeline reads them.

In [0]:
%sql
-- Order items: 900 rows across 300 orders
CREATE OR REPLACE TEMP VIEW v_raw_items AS
SELECT item_id,
       cast(rand() * 300 AS INT) + 1 AS order_id,
       cast(rand() * 50  AS INT) + 1 AS product_id,
       cast(rand() * 10  AS INT) + 1 AS quantity,
       round(rand() * 495 + 5, 2)    AS unit_price
FROM range(1, 901) t(item_id);

SELECT * FROM v_raw_items LIMIT 5;

item_id,order_id,product_id,quantity,unit_price
1,31,46,5,31.43
2,237,7,9,290.37
3,195,50,4,148.51
4,187,42,1,7.49
5,233,50,10,221.75


In [0]:
%sql
-- Orders: 300 rows
CREATE OR REPLACE TEMP VIEW v_raw_orders AS
SELECT o AS order_id,
       cast(rand() * 100 AS INT) + 1 AS customer_id,
       date_format(date_add(DATE'2025-01-01', cast(rand() * 178 AS INT)), 'yyyy-MM-dd') AS order_date,
       element_at(array('completed', 'pending', 'cancelled', 'shipped'), cast(rand() * 4 AS INT) + 1) AS status
FROM range(1, 301) t(o);

SELECT * FROM v_raw_orders LIMIT 5;

order_id,customer_id,order_date,status
1,23,2025-05-11,pending
2,8,2025-06-08,completed
3,54,2025-01-30,completed
4,52,2025-01-10,pending
5,48,2025-05-17,completed


In [0]:
%sql
-- Products: 50 rows
CREATE OR REPLACE TEMP VIEW v_raw_products AS
SELECT id AS product_id,
       concat('Product_', id) AS product_name,
       element_at(array('Electronics', 'Home', 'Sports', 'Toys', 'Books'), cast(rand() * 5 AS INT) + 1) AS category,
       round(rand() * 495 + 5, 2) AS price,
       cast(rand() * 20 AS INT) + 1 AS supplier_id
FROM range(1, 51) t(id);

SELECT * FROM v_raw_products LIMIT 5;

product_id,product_name,category,price,supplier_id
1,Product_1,Toys,108.07,11
2,Product_2,Sports,459.92,7
3,Product_3,Books,291.75,19
4,Product_4,Home,274.34,8
5,Product_5,Sports,32.41,20


In [0]:
%sql
-- Suppliers: 20 rows
CREATE OR REPLACE TEMP VIEW v_raw_suppliers AS
SELECT id AS supplier_id,
       concat('Supplier_', id) AS supplier_name,
       element_at(array('Israel', 'USA', 'Germany', 'China', 'Turkey'), cast(rand() * 5 AS INT) + 1) AS country,
       round(rand() * 3.9 + 1, 1) AS rating
FROM range(1, 21) t(id);

SELECT * FROM v_raw_suppliers LIMIT 5;

supplier_id,supplier_name,country,rating
1,Supplier_1,China,3.3
2,Supplier_2,Turkey,4.3
3,Supplier_3,Turkey,3.6
4,Supplier_4,Israel,2.6
5,Supplier_5,Israel,1.3


In [0]:
%sql
-- Customers: 100 rows (email is null for ~5% to simulate dirty raw data)
CREATE OR REPLACE TEMP VIEW v_raw_customers AS
SELECT id AS customer_id,
       concat('Customer_', id) AS customer_name,
       CASE WHEN rand() < 0.05 THEN NULL ELSE concat('customer', id, '@example.com') END AS email,
       element_at(array('Tel Aviv', 'Haifa', 'Jerusalem', 'Netanya', 'Beersheba'), cast(rand() * 5 AS INT) + 1) AS city,
       date_format(date_add(DATE'2024-01-01', cast(rand() * 364 AS INT)), 'yyyy-MM-dd') AS signup_date
FROM range(1, 101) t(id);

SELECT * FROM v_raw_customers LIMIT 5;

customer_id,customer_name,email,city,signup_date
1,Customer_1,customer1@example.com,Tel Aviv,2024-05-11
2,Customer_2,customer2@example.com,Tel Aviv,2024-03-30
3,Customer_3,null,Beersheba,2024-10-08
4,Customer_4,customer4@example.com,Haifa,2024-06-25
5,Customer_5,customer5@example.com,Jerusalem,2024-06-23


In [0]:
# Export all generated views as CSV files to the bronze volume
VOLUME_BASE = "/Volumes/ofer_demo/bronze/raw_csv"

views = {
    "raw_items": "v_raw_items",
    "raw_orders": "v_raw_orders",
    "raw_products": "v_raw_products",
    "raw_suppliers": "v_raw_suppliers",
    "raw_customers": "v_raw_customers",
}

for table_name, view_name in views.items():
    df = spark.table(view_name)
    (
        df.write
          .mode("overwrite")
          .option("header", "true")
          .csv(f"{VOLUME_BASE}/{table_name}")
    )
    print(f"{table_name}: {df.count()} rows -> {VOLUME_BASE}/{table_name}")

raw_items: 900 rows -> /Volumes/ofer_demo/bronze/raw_csv/raw_items
raw_orders: 300 rows -> /Volumes/ofer_demo/bronze/raw_csv/raw_orders
raw_products: 50 rows -> /Volumes/ofer_demo/bronze/raw_csv/raw_products
raw_suppliers: 20 rows -> /Volumes/ofer_demo/bronze/raw_csv/raw_suppliers
raw_customers: 100 rows -> /Volumes/ofer_demo/bronze/raw_csv/raw_customers
